In [9]:
import torch
import torch.optim as optim
from utils.SplitData import split_data_Train_Val_Test
from torch.utils.tensorboard import SummaryWriter # type: ignore

from model.trace import TRACE
#from dataset.otto_trace import TraceOttoDataSet
from dataset.otto_final import TraceOttoDataset
from utils.EarlyStopping import EarlyStopping
from utils.feature_engineering import get_between_features, get_elapsed_feature
from sklearn.metrics import f1_score,precision_score,recall_score

import os
import numpy as np
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [10]:
#DataSet    
dataset_processed = TraceOttoDataset(
        file_name='train.jsonl',
        input_seq_len=64,
        min_timestamps_per_sample=16,max_samples=2000)


print(dataset_processed[0])


({'session_id': tensor(0), 'aid': tensor([1517085, 1563459, 1309446,   16246, 1781822, 1152674, 1649869,  461689,
         305831,  461689,  362233, 1649869, 1649869,  984597, 1649869,  803544,
        1110941, 1190046, 1760685,  631008,  461689, 1190046, 1650637,  313546,
        1650637,  979517,  351157, 1062149, 1157384, 1841388, 1469630,  305831,
        1110548, 1110548,  305831, 1650114, 1604396, 1009750, 1800933,  495779,
         394655,  495779,  789245,  789245,  366890,  361317, 1700164, 1755597,
         789245,  784978, 1171505,  784978, 1700164,  784978, 1521766, 1725503,
         528847, 1816325,  984597, 1072782,  173702, 1072782, 1407538, 1629651]), 'timestamps': tensor([1659304800025, 1659304904511, 1659367439426, 1659367719997,
        1659367871344, 1659367885796, 1659369893840, 1659369898050,
        1659370027105, 1659370027105, 1659370064916, 1659370067686,
        1659371003682, 1659371033243, 1659371042297, 1659371044075,
        1659371104329, 1659371123063, 

In [ ]:
#Split the Data into Training_loader, Validation_loader and test_loaders
train_loader, validation_loader, test_loader = split_data_Train_Val_Test(dataset_processed)
    

In [ ]:
#calling the max aid and type for combating the Out of Range Error -> Learning Embeddings
max_aid = max(
        session[0]["aid"].max().item()
        for session in dataset_processed
    )
max_type = max(
        session[0]["type"].max().item()
        for session in dataset_processed
)
num_embeddings_aid = max_aid + 1  
num_embeddings_event_type = max_type + 1
    
#Initialize the Model TRACE(Model Architecture TRACE paper Part 2.3)
trace_model = TRACE(
    num_embeddings_aid=num_embeddings_aid,
    num_embeddings_event_type=num_embeddings_event_type,
    embedding_dim=32,
    num_classes=1
)

In [ ]:
"""
Shape of train_loader, val_loader, Test_loader
"""
for inputs_train, targets_train in train_loader:
    print("The version of train_loader")
    print(inputs_train.keys())
    print(targets_train.keys())
    print(
        f"Shape Aids: {inputs_train['aid'].shape}, "
        f"Shape Timestamps: {inputs_train['timestamps'].shape}, "
        f"Shape Type: {inputs_train['type'].shape}"
    )
    break  

for inputs_val, targets_val in validation_loader:
    
    print("The version of val_loader")
    print(inputs_val.keys())
    print(targets_val.keys())
    
    print(
        f"Shape Aids: {inputs_val['aid'].shape}, "
        f"Shape Timestamps: {inputs_val['timestamps'].shape}, "
        f"Shape Type: {inputs_val['type'].shape}"
    )
    break
for inputs_test, targets_test in test_loader:
    print("The version of train_loader")
    print(inputs_test.keys())
    print(targets_test.keys())
    print(
        f"Shape Aids: {inputs_test['aid'].shape}, "
        f"Shape Timestamps: {inputs_test['timestamps'].shape}, "
        f"Shape Type: {inputs_test['type'].shape}"
    )
    break

print("==================================SIZE=================================== ")
print(f"Train LOADER {len(train_loader)}")  
print(f"Validation LOADER {len(validation_loader)}")  
print(f"Test LOADER {len(test_loader)}")  


In [ ]:
trace_model = trace_model.to(device)
optimizer = optim.AdamW(trace_model.parameters(), lr=5e-5, weight_decay=1e-6)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimizer,cooldown=1,
                                                        mode="max",
                                                        factor=0.5,
                                                        patience=2,
                                                        min_lr=1e-6)
early_stopping = EarlyStopping(patience=7,
                                   min_delta=1e-4,
                                   mode="max",
                                   path=f"ModelTrace_MoreNeurons_lossWeighted4_1_1_2026_CheckPoint.pt")
    
#Summary Writer for tensorBoard
tensor_board_writer = SummaryWriter(log_dir=f"runs/Testing_HyperparameterTuning_1/01/2026_version1_MoreNeurons_lossWeighted4")

In [ ]:
print("Started the Training")
#Figthing Data Imbalanced
labels_list = []
for inputs, targets in train_loader:
    labels_list.append(targets["PD1"].view(-1)) #(Batch, )
        
labels = torch.cat(labels_list, dim=0) #(N, )           
    
#Number of positives in the train_loader
num_pos = (labels == 1).sum().item()
    
#Number of Negatives in the train_loader
num_neg = (labels == 0).sum().item()
    
ratio = num_neg / max(num_pos, 1)
    
weights = torch.tensor([ratio], device=device)
    
print("Train pos/neg:", num_pos, num_neg, "pos_weight:", weights.item())
    
#Compute pos_weight = (#neg / #pos) for BCEWithLogitsLoss to handle class imbalance (training only)
criterion_train = torch.nn.BCEWithLogitsLoss(pos_weight=weights) 
    
criterion_validation = torch.nn.BCEWithLogitsLoss()
#Learning Rate Scheduler
#To Save the Best F1 for the Model
best_val_f1 = -1.0
best_global_thr = 0.5
    
num_epochs = 40
    
for epoch in range(num_epochs):
        
    #F1 Score training
    all_train_y_true = []
    all_train_y_pred = []
            
    #F1 Score validation
    all_val_y_true = []
            
            
    # -------------------------------TRAINING ---------------------------
    #Initializing the training variables
    trace_model.train()
    epoch_loss = 0.0
    correct_train_PD1 = 0
    total_train_PD1 = 0
            
    for inputs_train, targets_train in train_loader:
                
        label_train_PD1 = targets_train["PD1"].unsqueeze(1).to(device)
        #Changing the Inputs -> to have GPU for JupyterGPUHub
        inputs_train = {
            k: v.to(device)
                for k, v in inputs_train.items()
            }
        #Calculation of the timestamps(Feature Engineer Trace Paper Part 2.2)
        delta_elapsed = get_elapsed_feature(inputs_train["timestamps"]).to(device)
        delta_between = get_between_features(inputs_train["timestamps"]).to(device)
                
        optimizer.zero_grad(set_to_none=True)
        #Predictions of the model
        logits_train = trace_model(
                inputs_train["aid"],
                inputs_train["type"],
                delta_elapsed,
                delta_between
            )
                
        #Calculation loss for Training using BCEWithLogitsLoss
        loss_training = criterion_train(logits_train,label_train_PD1.float())
        loss_training.backward()
        optimizer.step()
                
        epoch_loss += loss_training.item()
                
        # ============ PD1(Prediction, calculation of Accuracy) ============
        probs_PD1 = torch.sigmoid(logits_train)
        preds_PD1 = (probs_PD1 >= 0.5).float()
        correct_train_PD1 += (preds_PD1 == label_train_PD1).sum().item()
        total_train_PD1 += label_train_PD1.numel()
                
        #F1 Score For training 
        all_train_y_true.append(label_train_PD1.detach().cpu())
        all_train_y_pred.append(preds_PD1.detach().cpu())
    
    #Training Loss and Accuracy 
    train_loss = epoch_loss / len(train_loader)
    train_acc_PD1 = correct_train_PD1 / max(total_train_PD1, 1)
            
    #F1 Score for training
    all_train_y_true = torch.cat(all_train_y_true).numpy().ravel()
    all_train_y_pred = torch.cat(all_train_y_pred).numpy().ravel()
    train_f1_PD1 = f1_score(all_train_y_true, all_train_y_pred, zero_division=0)
            
    #TensorBoard Writing
    tensor_board_writer.add_scalar("Train/F1_PD1", train_f1_PD1, epoch)
    tensor_board_writer.add_scalar("Training/Loss", train_loss, epoch)
    tensor_board_writer.add_scalar("Train/Acc_PD1", train_acc_PD1, epoch)
            
    
    
    # -------------------------------VALIDATION---------------------------
    #Initializing the validation variables    
    trace_model.eval()
    val_loss = 0.0
    correct_val_PD1 = 0
    total_val_PD1 = 0
            
    all_val_probs = []
    all_val_y_true = []
    
    with torch.no_grad():
        for inputs_val, targets_val in validation_loader:
            label_val_PD1 = targets_val["PD1"].unsqueeze(1).to(device)
    
            inputs_val = {k: v.to(device) for k, v in inputs_val.items()}
    
            delta_elapsed = get_elapsed_feature(inputs_val["timestamps"]).to(device)
            delta_between = get_between_features(inputs_val["timestamps"]).to(device)
    
            logits_val = trace_model(
                inputs_val["aid"],
                inputs_val["type"],
                delta_elapsed,
                delta_between
            )# -> (Batch, number_of_classes)
             
            #Calculation of validation loss    
            loss_validation = criterion_validation(logits_val, label_val_PD1.float())
            val_loss += loss_validation.item()

            #Logits converted to sigmoid
            probs_PD1_val = torch.sigmoid(logits_val) # Convert logits to probabilities in [0, 1]
            preds_PD1_val = (probs_PD1_val >= 0.5).float()
            correct_val_PD1 += (preds_PD1_val == label_val_PD1).sum().item()
            total_val_PD1 += label_val_PD1.numel()
    
            #Save the Global F1
            all_val_y_true.append(label_val_PD1.detach().cpu()) #breaks graph and prepare for numpy / sklearn
            all_val_probs.append(probs_PD1_val.detach().cpu())
    
    # ----Concatonate the Probabilities and true labels ----
    all_val_y_true = torch.cat(all_val_y_true).numpy().ravel()
    all_val_probs = torch.cat(all_val_probs).numpy().ravel()
    
    #Searching for the right threshold from (0.1 -> 0.9) tries 81 treshold
    thresholds = np.linspace(0.1, 0.9, 81)
    
    #Initializtion of variables threshold and f1
    best_thr = 0.5
    best_f1 = 0.0
    
    for t in thresholds:
        preds_thr = (all_val_probs >= t).astype(int)
        f1 = f1_score(all_val_y_true, preds_thr, zero_division=0)
        
        #Saves the best threshold and F1 score of the Epoch
        if f1 > best_f1:
            best_f1 = f1
            best_thr = t
        
    #The best threshold and F1 score of the Epoch
    val_f1_PD1 = best_f1
    threshold = best_thr
    
    #Making predictions with the best threshold
    val_pred = (all_val_probs >= threshold).astype(int)
    val_precision = precision_score(all_val_y_true, val_pred, zero_division=0)
    val_recall = recall_score(all_val_y_true, val_pred, zero_division=0)
      
    
# Keep the best threshold for this epoch, and also the best threshold seen across all epochs   
    if val_f1_PD1 > best_val_f1:
        best_val_f1 = val_f1_PD1
        best_global_thr = threshold
    
        
    #Validation Loss and Accuracy 
    val_loss /= len(validation_loader)
    #val_acc_PD1 = correct_val_PD1 / max(total_val_PD1, 1)
    
    #The best accurancy with the optimal threshold
    val_acc_best_thr = ((all_val_probs >= threshold).astype(int) == all_val_y_true.astype(int)).mean()
    
    #TensorBoard
    tensor_board_writer.add_scalar("Val/Loss", val_loss, epoch)
    tensor_board_writer.add_scalar("Val/Acc_PD1_best_thr", val_acc_best_thr, epoch)
    tensor_board_writer.add_scalar("Val/F1_PD1", val_f1_PD1, epoch)
    tensor_board_writer.add_scalar("Val/Best_Threshold", threshold, epoch)
    tensor_board_writer.add_scalar("Val/Best_Global_Threshold", best_global_thr, epoch)
    tensor_board_writer.add_scalar("Val/Precision_PD1", val_precision, epoch)
    tensor_board_writer.add_scalar("Val/Recall_PD1", val_recall, epoch)

    lr_scheduler.step(val_f1_PD1)
                            
    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc_PD1:.4f} | Train F1: {train_f1_PD1:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val F1: {val_f1_PD1:.4f} | "
        f"BestThr: {threshold:.3f} | Val Acc: {val_acc_best_thr:.4f} "
        f"Val Precision: {val_precision} | Val Recall {val_recall} "
    )

        #Print the Current Learning rate after the Lr
    current_lr = optimizer.param_groups[0]["lr"]
    tensor_board_writer.add_scalar("LR", current_lr, epoch)
    print(f"This is the LR: {current_lr}")
            
    #Early Stopping
    early_stopping(val_f1_PD1, trace_model)
    if early_stopping.early_stop:
        print("Early stopping triggered")
        break
        
tensor_board_writer.close()

In [ ]:
#Saves the Model CheckPoint if the JupyterGpuHub the session expires
early_stopping.load_best_weights(trace_model)
    
#Save the total Model after the training
torch.save({
        "model_state_dict": trace_model.state_dict(),
        "best_val_f1": best_val_f1,
        "best_global_threshold": best_global_thr,
    }, "ModelTrace_MoreNeurons_lossWeighted4_version_1_1_2026.pt")
